# AnyCam — 03 · Step 5: baseline training (normal model, all 204 bands)

Trains a ResNet18 (ImageNet weights, first layer widened to 204 bands) on HyperLeaf2024 and tests it
on the official test plots. Expected: cultivar accuracy close to the HyperLeaf paper (~0.89).

**Before running (right panel):**
1. **Add Input** → Competitions → **HyperLeaf2024**
2. **Settings → Accelerator → GPU T4 x2** (or P100)
3. **Settings → Internet → On**
4. Optional: Add-ons → Secrets → `GITHUB_WRITE_TOKEN` (On) to upload results to GitHub automatically

Then **Run All** (about 10-20 minutes).

In [ ]:
import os, subprocess, glob, json, shutil, pathlib
REPO = 'https://github.com/ShahnawazQ007/one-model-many-camera.git'
if not os.path.exists('/kaggle/working/anycam'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, '/kaggle/working/anycam'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/anycam', 'pull', '-q'], check=True)
os.chdir('/kaggle/working/anycam')
!pip -q install timm tifffile imagecodecs
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - turn on the GPU accelerator!')

In [ ]:
# find the HyperLeaf data folder (searches everything under /kaggle/input)
hits = [os.path.dirname(f) for f in glob.glob('/kaggle/input/**/train.csv', recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(f), 'images'))]
if not hits:
    print('Inputs attached to this notebook:')
    for d in glob.glob('/kaggle/input/**/', recursive=True)[:30]: print('  ', d)
    raise SystemExit('HyperLeaf not found: in the right panel, Add Input -> search hyperleaf -> the one with the trophy icon (+)')
DATA = hits[0]
print('data:', DATA, '| images:', len(glob.glob(f'{DATA}/images/*.tiff')))


In [ ]:
# Step 5a: official competition split (train plots vs test plots)
!python experiments/train_baseline.py --data {DATA} --split official --epochs 40 --workers 4 --out runs/baseline_official

In [ ]:
# Step 5b (optional, ~10-15 min more): first fold of the plot-grouped cross-validation
!python experiments/train_baseline.py --data {DATA} --split cv --seed 0 --fold 0 --epochs 40 --workers 4 --out runs/baseline_cv_s0_f0

In [ ]:
# figures: training curves, confusion matrix, per-cultivar accuracy, predicted vs true for each trait
!python scripts/plot_run.py runs/*
from IPython.display import Image, display
for f in sorted(glob.glob('runs/*/figures.png')):
    print(f); display(Image(f))


In [ ]:
# collect the small result files (no model weights) and upload them to GitHub results/ if the token is set
res = pathlib.Path('/kaggle/working/step05_results'); res.mkdir(exist_ok=True)
for run in glob.glob('runs/*'):
    dst = res / pathlib.Path(run).name; dst.mkdir(exist_ok=True)
    for f in ['metrics.json', 'predictions.csv', 'history.csv', 'config.json', 'band_norm.json', 'figures.png']:
        if os.path.exists(f'{run}/{f}'): shutil.copy(f'{run}/{f}', dst)
    print(pathlib.Path(run).name, json.dumps(json.load(open(f'{run}/metrics.json'))['cultivar']))
subprocess.run(['python', 'scripts/push_results.py', str(res), 'step05_baseline'])

## Done
If the last cell says **Uploaded to GitHub**, tell Claude "done". Otherwise download the folder
`/kaggle/working/step05_results` from the Output tab and send it to Claude.